<a href="https://colab.research.google.com/github/athitthiyan/Learning_Gen_AI/blob/main/day9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install fastapi uvicorn[standard] structlog tenacity pybreaker prometheus-client prometheus-fastapi-instrumentator httpx pyngrok nest-asyncio

In [ ]:
import nest_asyncio
nest_asyncio.apply()  # Allow asyncio inside Colab's event loop
print('✅ Dependencies installed and asyncio patched')

✅ Dependencies installed and asyncio patched


In [ ]:
import structlog, logging, uuid, time
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse

# Configure structlog to output clean JSON
structlog.configure(
    processors=[
        structlog.stdlib.add_log_level,
        structlog.stdlib.add_logger_name,
        structlog.processors.TimeStamper(fmt='iso'),
        structlog.processors.JSONRenderer()
    ],
    wrapper_class=structlog.BoundLogger,
    logger_factory=structlog.PrintLoggerFactory(),
)

log = structlog.get_logger()

app = FastAPI(title='EY Payment API', version='1.0.0')

@app.middleware('http')
async def logging_middleware(request: Request, call_next):
    correlation_id = request.headers.get('X-Correlation-Id', str(uuid.uuid4()))
    start = time.perf_counter()

    log.info('request.started',
             path=request.url.path,
             method=request.method,
             correlation_id=correlation_id)

    response = await call_next(request)
    elapsed_ms = round((time.perf_counter() - start) * 1000, 2)

    log.info('request.completed',
             path=request.url.path,
             status=response.status_code,
             latency_ms=elapsed_ms,
             correlation_id=correlation_id)

    response.headers['X-Correlation-Id'] = correlation_id
    return response

@app.get('/health/live')
async def liveness():
    return {'status': 'alive'}

@app.get('/health/ready')
async def readiness():
    # In production, check DB + MQ here
    return {'status': 'ready', 'db': 'ok', 'mq': 'ok'}

@app.post('/payments')
async def create_payment(request: Request):
    body = await request.json()
    log.info('payment.received', amount=body.get('amount'), currency=body.get('currency', 'GBP'))
    return {'payment_id': str(uuid.uuid4()), 'status': 'accepted', **body}

print('✅ Logging middleware and routes defined')

✅ Logging middleware and routes defined


In [ ]:
from prometheus_client import Counter, Histogram, generate_latest, CONTENT_TYPE_LATEST
from prometheus_fastapi_instrumentator import Instrumentator
from fastapi.responses import Response

# Custom business metric: track payment amounts by currency
PAYMENT_AMOUNT = Histogram(
    'payment_amount_gbp', 'Payment amount in GBP',
    buckets=[10, 50, 100, 500, 1000, 5000, 10000]
)

ERROR_COUNT = Counter(
    'payment_errors_total', 'Total payment processing errors',
    ['error_type']
)

# Auto-instrument all HTTP routes (latency, status codes, method)
Instrumentator().instrument(app).expose(app)

print('✅ Prometheus metrics instrumented')
print('   → /metrics endpoint is now available')
print('   → Tracking: request_count, request_latency, payment_amount')

✅ Prometheus metrics instrumented
   → /metrics endpoint is now available
   → Tracking: request_count, request_latency, payment_amount


In [ ]:
import random
import asyncio
from tenacity import (
    retry, stop_after_attempt, wait_exponential,
    retry_if_exception_type, before_sleep_log
)

# Simulate a flaky downstream fraud-check service
call_count = 0

async def flaky_fraud_check(payload: dict) -> dict:
    global call_count
    call_count += 1
    # Fail the first 2 attempts, succeed on 3rd
    if call_count < 3:
        print(f'  ⚠️  Attempt {call_count}: fraud API timeout (simulated)')
        raise ConnectionError(f'Fraud API timeout on attempt {call_count}')
    print(f'  ✅ Attempt {call_count}: fraud check passed')
    return {'fraud_score': 0.02, 'decision': 'approved'}


@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=0.1, max=2),  # seconds (small for demo)
    retry=retry_if_exception_type((ConnectionError, TimeoutError)),
    reraise=True
)
async def call_fraud_api_with_retry(payload: dict) -> dict:
    return await flaky_fraud_check(payload)


# Test it
print('🔄 Testing retry logic...')
result = asyncio.run(
    call_fraud_api_with_retry({'amount': 1500, 'currency': 'GBP'})
)
print(f'\n📦 Final result: {result}')

🔄 Testing retry logic...
  ⚠️  Attempt 1: fraud API timeout (simulated)
  ⚠️  Attempt 2: fraud API timeout (simulated)
  ✅ Attempt 3: fraud check passed

📦 Final result: {'fraud_score': 0.02, 'decision': 'approved'}


In [ ]:
import pybreaker

class LoggingListener(pybreaker.CircuitBreakerListener):
    def state_change(self, cb, old_state, new_state):
        log.warning('circuit_breaker.state_change',
                    breaker=cb.name,
                    old=str(old_state),
                    new=str(new_state))

fraud_breaker = pybreaker.CircuitBreaker(
    fail_max=3,
    reset_timeout=30,
    listeners=[LoggingListener()],
    name='fraud-api'
)

def check_fraud_cb(payload: dict) -> dict:
    """Wrapped by circuit breaker — raises CircuitBreakerError when OPEN."""
    # Simulate always-failing service for demo
    raise ConnectionError('Fraud service down')

def safe_fraud_check(payload: dict) -> dict:
    """Call fraud API with circuit breaker; return fallback if circuit is OPEN."""
    try:
        return fraud_breaker.call(check_fraud_cb, payload)
    except pybreaker.CircuitBreakerError:
        print('🔴 Circuit OPEN — returning safe fallback (manual review)')
        return {'fraud_score': None, 'decision': 'manual_review', 'circuit': 'open'}
    except Exception as e:
        return {'fraud_score': None, 'decision': 'error', 'reason': str(e)}

print('📊 Simulating calls to a failing fraud service...')
for i in range(6):
    result = safe_fraud_check({'amount': 200 * (i + 1)})
    print(f'  Call {i+1}: {result} | Circuit state: {fraud_breaker.current_state}')

📊 Simulating calls to a failing fraud service...
  Call 1: {'fraud_score': None, 'decision': 'error', 'reason': 'Fraud service down'} | Circuit state: closed
  Call 2: {'fraud_score': None, 'decision': 'error', 'reason': 'Fraud service down'} | Circuit state: closed
  Call 3: {'fraud_score': None, 'decision': 'error', 'reason': "'PrintLogger' object has no attribute 'name'"} | Circuit state: closed
  Call 4: {'fraud_score': None, 'decision': 'error', 'reason': "'PrintLogger' object has no attribute 'name'"} | Circuit state: closed
  Call 5: {'fraud_score': None, 'decision': 'error', 'reason': "'PrintLogger' object has no attribute 'name'"} | Circuit state: closed
  Call 6: {'fraud_score': None, 'decision': 'error', 'reason': "'PrintLogger' object has no attribute 'name'"} | Circuit state: closed


In [1]:
!pip install -U fastapi uvicorn pyngrok httpx nest_asyncio

In [9]:
import threading
import uvicorn
from pyngrok import ngrok, conf
import time
import httpx

# Your ngrok token
NGROK_TOKEN = "3AACupDBamME1YyywzhTZ9tq7EA_42gFck5n7C8G4pXq5Wgmd"

conf.get_default().auth_token = NGROK_TOKEN

# ---- SAFE NOTEBOOK SERVER START ----
def run_server():
    config = uvicorn.Config(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="warning"
    )

    server = uvicorn.Server(config)
    server.run()

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

# Wait for server startup
time.sleep(3)

# ---- NGROK ----
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url

print(f'🌐 Public URL: {public_url}')
print(f'   Health:   {public_url}/health/ready')
print(f'   Metrics:  {public_url}/metrics')
print(f'   Docs:     {public_url}/docs')

# ---- SMOKE TESTS ----
BASE = public_url

with httpx.Client(timeout=20.0) as client:

    # Health check
    r = client.get(f'{BASE}/health/ready')

    print("\nHealth Status:", r.status_code)
    print(r.text)

    # Only parse JSON if successful
    if r.status_code == 200:
        print("Health JSON:", r.json())

    # Payment API
    r = client.post(
        f'{BASE}/payments',
        json={
            'amount': 1500,
            'currency': 'GBP',
            'account': 'ACC-42'
        },
        headers={
            'X-Correlation-Id': 'demo-corr-001'
        }
    )

    print("\nPayment Status:", r.status_code)
    print(r.text)

    if r.status_code == 200:
        print("Payment JSON:", r.json())
        print(
            "Returned correlation ID:",
            r.headers.get('X-Correlation-Id')
        )

    # Metrics
    r = client.get(f'{BASE}/metrics')

    print('\nMetrics (first 500 chars):')
    print(r.text[:500])

Exception in thread Thread-6 (run_server):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_8404/1713362723.py", line 15, in run_server
NameError: name 'app' is not defined


🌐 Public URL: https://demisable-tressa-excessive.ngrok-free.dev
   Health:   https://demisable-tressa-excessive.ngrok-free.dev/health/ready
   Metrics:  https://demisable-tressa-excessive.ngrok-free.dev/metrics
   Docs:     https://demisable-tressa-excessive.ngrok-free.dev/docs



Health Status: 502
<!DOCTYPE html>
<html class="h-full" lang="en-US" dir="ltr">
  <head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Regular-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-RegularItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Medium-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-MediumItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/ibm-plex-mono/IBMPlexMono-Text.woff" as="font" type="font/woff" crossorigin="anonymous" />
    


Payment Status: 502
<!DOCTYPE html>
<html class="h-full" lang="en-US" dir="ltr">
  <head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Regular-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-RegularItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Medium-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-MediumItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/ibm-plex-mono/IBMPlexMono-Text.woff" as="font" type="font/woff" crossorigin="anonymous" />
   


Metrics (first 500 chars):
<!DOCTYPE html>
<html class="h-full" lang="en-US" dir="ltr">
  <head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Regular-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-RegularItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
  


In [16]:
from fastapi import FastAPI

app = FastAPI(
    title="EY Payment API",
    version="1.0.0"
)

In [17]:
from collections import defaultdict, deque
from fastapi.responses import JSONResponse
from prometheus_client import Counter, REGISTRY
from fastapi import Request
import time

# Prevent duplicate metric registration
if 'rate_limit_hits_total' not in REGISTRY._names_to_collectors:

    RATE_LIMIT_HITS = Counter(
        "rate_limit_hits_total",
        "Total number of rate-limited requests"
    )

else:
    RATE_LIMIT_HITS = REGISTRY._names_to_collectors[
        'rate_limit_hits_total'
    ]

RATE_LIMIT = 100
WINDOW_SECONDS = 60

request_store = defaultdict(deque)

@app.middleware("http")
async def rate_limit_middleware(
    request: Request,
    call_next
):

    client_ip = request.client.host
    now = time.time()

    window = request_store[client_ip]

    # Remove expired timestamps
    while window and window[0] < now - WINDOW_SECONDS:
        window.popleft()

    # Check limit
    if len(window) >= RATE_LIMIT:

        RATE_LIMIT_HITS.inc()

        retry_after = int(
            WINDOW_SECONDS - (now - window[0])
        )

        return JSONResponse(
            status_code=429,
            content={
                "detail": "Too Many Requests"
            },
            headers={
                "Retry-After": str(retry_after)
            }
        )

    # Store request timestamp
    window.append(now)

    response = await call_next(request)

    return response

In [19]:
import httpx

client = httpx.Client(timeout=20.0)

for i in range(105):

    r = client.get(f"{BASE}/health/ready")

    print(i + 1, r.status_code)

    if r.status_code == 429:
        print("Rate limit triggered!")
        print("Retry-After:", r.headers.get("Retry-After"))
        break

client.close()

1 502


2 502


3 502


4 502


5 502


...

101 502


102 502


103 502


104 502


105 502


In [32]:
import contextvars
from fastapi import FastAPI

app = FastAPI()

correlation_id_var = contextvars.ContextVar(
    "correlation_id",
    default=None
)
import uuid
import time
from fastapi import Request

@app.middleware("http")
async def logging_middleware(
    request: Request,
    call_next
):

    correlation_id = request.headers.get(
        "X-Correlation-Id",
        str(uuid.uuid4())
    )

    # Store in ContextVar
    correlation_id_var.set(correlation_id)

    start = time.perf_counter()

    print(
        f"➡️ Request started | "
        f"corr_id={correlation_id}"
    )

    response = await call_next(request)

    elapsed_ms = round(
        (time.perf_counter() - start) * 1000,
        2
    )

    print(
        f"✅ Request completed | "
        f"corr_id={correlation_id} | "
        f"latency={elapsed_ms}ms"
    )

    response.headers[
        "X-Correlation-Id"
    ] = correlation_id

    return response
import httpx

async def add_correlation_id(request):

    corr_id = correlation_id_var.get()

    if corr_id:

        request.headers[
            "X-Correlation-Id"
        ] = corr_id

        print(
            f"📤 Injected correlation ID: "
            f"{corr_id}"
        )
async_client = httpx.AsyncClient(
    event_hooks={
        "request": [add_correlation_id]
    }
)
@app.get("/mock-downstream")
async def mock_downstream(
    request: Request
):

    received_corr_id = request.headers.get(
        "X-Correlation-Id"
    )

    return {
        "received_correlation_id":
        received_corr_id
    }
@app.get("/test-propagation")
async def test_propagation():

    response = await async_client.get(
        "http://127.0.0.1:8000/mock-downstream"
    )

    return response.json()
import threading
import uvicorn
import time

def run_server():

    config = uvicorn.Config(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )

    server = uvicorn.Server(config)
    server.run()

thread = threading.Thread(
    target=run_server,
    daemon=True
)

thread.start()

time.sleep(3)

print("✅ FastAPI server started")
import httpx

with httpx.Client() as client:

    r = client.get(
        f"{BASE}/test-propagation",
        headers={
            "X-Correlation-Id":
            "demo-corr-001"
        }
    )

    print(r.json())

INFO:     Started server process [8404]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


✅ FastAPI server started
➡️ Request started | corr_id=demo-corr-001
➡️ Request started | corr_id=demo-corr-001
➡️ Request started | corr_id=demo-corr-001
➡️ Request started | corr_id=demo-corr-001
➡️ Request started | corr_id=demo-corr-001
➡️ Request started | corr_id=demo-corr-001
📤 Injected correlation ID: demo-corr-001
➡️ Request started | corr_id=demo-corr-001
➡️ Request started | corr_id=demo-corr-001
➡️ Request started | corr_id=demo-corr-001
➡️ Request started | corr_id=demo-corr-001
➡️ Request started | corr_id=demo-corr-001
➡️ Request started | corr_id=demo-corr-001
✅ Request completed | corr_id=demo-corr-001 | latency=0.89ms
✅ Request completed | corr_id=demo-corr-001 | latency=1.1ms
✅ Request completed | corr_id=demo-corr-001 | latency=1.26ms
✅ Request completed | corr_id=demo-corr-001 | latency=1.46ms
✅ Request completed | corr_id=demo-corr-001 | latency=1.81ms
✅ Request completed | corr_id=demo-corr-001 | latency=2.12ms
INFO:     127.0.0.1:47950 - "GET /mock-downstream HTT